In [29]:
import pandas as pd
from regions import RectangleSkyRegion,CompoundSkyRegion, Regions
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np

from lightcurvelynx.obstable.ztf_obstable import ZTFObsTable
from lightcurvelynx.astro_utils.detector_footprint import DetectorFootprint
from mocpy import MOC


In [7]:
obs_log = pd.read_parquet('data/ztf_observing_log_combined_w_metadata.parquet')
colmap = {"ra":"ra",
          "dec":"dec",
          "time":"mjd",
          "zp":"zp_nJy",
          "filter":"filter",
          "sky":"sky_adu_ztfsn",
         }

In [8]:
#ztf ccd size 6144 × 6160 pixel * 16
pixel_scale = 1.01 #arcsec/pixel
center = SkyCoord(ra=0.0, dec=0.0, unit="deg", frame="icrs")

rect_region = RectangleSkyRegion(center=center, width=4*6144.* pixel_scale * u.arcsec, 
                                 height=4*6160.* pixel_scale * u.arcsec, angle=0.0 * u.deg)
ztf_fp1 = DetectorFootprint(rect_region, pixel_scale=pixel_scale)

ztf_obstable1 = ZTFObsTable(obs_log,colmap=colmap,detector_footprint=ztf_fp1)


In [17]:
#field dimension is 7.323 deg * 7.504 deg Dekany 2020 Table 3
pixel_scale = 1.01 #arcsec/pixel
center = SkyCoord(ra=0.0, dec=0.0, unit="deg", frame="icrs")

rect_region = RectangleSkyRegion(center=center, width=7.323 * u.deg, 
                                 height=7.504 * u.deg, angle=0.0 * u.deg)
ztf_fp2 = DetectorFootprint(rect_region, pixel_scale=pixel_scale)

ztf_obstable2 = ZTFObsTable(obs_log,colmap=colmap,detector_footprint=ztf_fp2)

In [10]:
sky_coverage1 = ztf_obstable1.estimate_coverage(use_footprint=True)

Evaluating Region: 100%|███████████████████| 1293/1293 [00:04<00:00, 286.00it/s]


In [18]:
sky_coverage2 = ztf_obstable2.estimate_coverage(use_footprint=True)

Evaluating Region: 100%|███████████████████| 1293/1293 [00:04<00:00, 267.25it/s]


In [38]:
sky_coverage1

31989.57209855318

In [15]:
(sky_coverage2* 0.867)/sky_coverage1

0.8697605728614403

In [39]:
sky_coverage2 ,sky_coverage2 * 0.867

(32133.31069597602, 27859.58037341121)

In [36]:
from mocpy import MOC
from regions import RectangleSkyRegion, Regions
from astropy.coordinates import SkyCoord
from lightcurvelynx.astro_utils.coordinate_utils import dedup_coords
import astropy.units as u
import numpy as np
from tqdm import tqdm

pixel_scale = 1.01  # arcsec/pixel

# Single CCD on-sky size
ccd_w_arcsec = 6144 * pixel_scale
ccd_h_arcsec = 6160 * pixel_scale

# Gaps from Dekany et al. 2020, Table 3
gap_ew_arcsec = 0.205 * 3600
gap_ns_arcsec = 0.140 * 3600

# CCD center offsets relative to pointing center (in arcsec)
def ccd_center_offsets(ccd_size, gap, n=4):
    step = ccd_size + gap
    return [(i - 1.5) * step for i in range(n)]

offsets_ew = ccd_center_offsets(ccd_w_arcsec, gap_ew_arcsec)
offsets_ns = ccd_center_offsets(ccd_h_arcsec, gap_ns_arcsec)

# Get unique pointings from your obs table
ra = ztf_obstable._table["ra"].to_numpy()
dec = ztf_obstable._table["dec"].to_numpy()
ra_uniq, dec_uniq, _ = dedup_coords(ra, dec, threshold=100.0 / 3600.0)
print(f"Building MOC from {len(ra_uniq)} unique pointings × 16 CCDs")

# Build MOC: one RectangleSkyRegion per CCD per pointing
max_depth = 10  # start low for speed, increase later if needed
moc = MOC.new_empty(max_depth=max_depth)

for p_ra, p_dec in tqdm(zip(ra_uniq, dec_uniq), total=len(ra_uniq)):
    cos_dec = np.cos(np.radians(p_dec))
    
    for dra in offsets_ew:
        for ddec in offsets_ns:
            ccd_dec = p_dec + ddec / 3600.0
            
            # Skip CCDs that fall off the pole
            if ccd_dec > 89.5 or ccd_dec < -89.5:
                continue
            
            ccd_ra = p_ra + dra / 3600.0 / cos_dec
            
            ccd_region = RectangleSkyRegion(
                center=SkyCoord(ra=ccd_ra, dec=ccd_dec, unit="deg", frame="icrs"),
                width=ccd_w_arcsec * u.arcsec,
                height=ccd_h_arcsec * u.arcsec,
                angle=0.0 * u.deg,
            )
            ccd_moc = MOC.from_astropy_regions(ccd_region, max_depth=max_depth)
            moc = moc.union(ccd_moc)

coverage = moc.sky_fraction * 41253.0
print(f"Coverage (with gaps): {coverage:.1f} deg²")

Building MOC from 1293 unique pointings × 16 CCDs


100%|███████████████████████████████████████| 1293/1293 [00:13<00:00, 93.10it/s]

Coverage (with gaps): 31862.6 deg²
